In [44]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

In [45]:
base_dir = Path(".").resolve()
data_dir = base_dir / "data"

In [46]:
# Distribution Rules

with open(data_dir / "metadata.json") as meta_file:
    metadata = json.load(meta_file)

with open(data_dir / "distribution_rules.json") as file:
    data = json.load(file, object_hook=lambda obj: {**obj.pop("lvl", {}), **obj})
    rules = data.get("rules", [])

    for rule in rules:
        rrg = rule.get("rrg")
        if rrg is None:
            rule["rrg"] = {}

        for key in list(rule.keys()):
            if key in metadata:
                rule[metadata[key]] = rule.pop(key)

        level_mapping = metadata["level_mapping"]
        for key, value in rule.items():
            if isinstance(value, dict):
                for k in list(value.keys()):
                    if k in level_mapping:
                        value[level_mapping[k]] = value.pop(k)

df_rules = pd.json_normalize(rules, sep="_")

In [48]:
df_rules[["id", "credential_list"]].head()

,id,credential_list
0,394291,"[5226, 8550]"
1,432673,None
2,498300,"[11807, 5318]"
3,394034,"[10206, 8456]"
4,394446,"[5997, 10901, 32713, 5129, 5226, 30830, 31318,..."


In [49]:
rules_client_credential = df_rules.explode("credential_list")[
    ["id", "credential_choices", "credential_list"]
]

In [50]:
rules_client_credential.head()

,id,credential_choices,credential_list
0,394291,1,5226
0,394291,1,8550
1,432673,0,None
2,498300,1,11807
2,498300,1,5318


In [ ]:
df_exploded["client_list"] = df_exploded["client_list"].astype(pd.Int64Dtype())

In [ ]:
print(df_exploded.client_list.unique())

In [ ]:
print(df_exploded[["id", "client_list"]])

In [ ]:
df_exploded[df_exploded["client_list"] == 11653].shape[0]

In [ ]:
df_exploded.to_csv("output.csv", index=False)

In [ ]:
df_exploded["client_choices"].unique()

In [ ]:
df["client_list"] = df["client_list"].apply(lambda x: [] if x is None else x)
# df["client_list"] = df["client_list"].apply(lambda x: [] if x is None else [int(i) for i in x])
# df["provider_values"] = df["provider_values"].apply(lambda x: [] if x is None else x)

In [ ]:
def count_rules(client_id, df):
    count = 0
    for rule in df["client_list"]:
        if str(client_id) in rule:
            count += 1
    return count


client_id = 11653
rule_count = count_rules(client_id, df)
print(f"Client {client_id} has {rule_count} applicable rules.")

In [ ]:
# df["hotel_list"] = df["hotel_list"].apply(lambda x: [] if x is None else x)

# df["hotel_list"] = df["hotel_list"].apply(
#     lambda x: [] if x is None else [i for i in x if i is not None]
# )

In [ ]:
mask = df.hotel_list.map(set([None]).issubset)

In [ ]:
df_hotel_contains_none = df[mask]

In [ ]:
df_hotel_contains_none[["id", "hotel_list"]]

In [ ]:
df_rules.columns

In [ ]:
mask_nautalia = (df["credential_choices"] == 2) & (
    df["credential_list"].apply(lambda x: 4 in x)
)
df_rules_nautalia = df_rules[mask_nautalia]

In [51]:
df_rules_nautalia = rules_client_credential.query(
    "credential_choices == 2 and credential_list == 4"
)

In [53]:
df_rules_nautalia.head()

,id,credential_choices,credential_list


In [58]:
rules_client_credential.loc[
    (rules_client_credential["credential_choices"] == 2)
    & (rules_client_credential["credential_list"] == str(4))
]

,id,credential_choices,credential_list
780,446223,2,4
1174,395719,2,4
